# Import Libraries

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import pickle

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

In [7]:
# Load dataset
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/StudentPerformance/data/raw/student_dataset_10000_rows.csv')

## 1. Create X and y features

In [8]:
X = df[[col for col in df.columns if col != "placement_status"]]
y = df['placement_status']

## 2. Train-test split

In [9]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=0.20,random_state=42,stratify=y)

print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


Training Samples: 8000
Testing Samples: 2000


## 3. Feature Engineering `cat and num columns`

In [10]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [11]:
print("\nNumerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)


Numerical Columns:
['study_hours', 'attendance', 'sleep_hours', 'internet_usage', 'assignments_completed', 'previous_score', 'exam_score']

Categorical Columns:
[]


###  Numerical preprocessing:
1. Fill missing values with median
2. Scale data to standard normal distribution

In [14]:
num_pipeline = Pipeline([
    ( "imputer",
        SimpleImputer(strategy="median")),

("scaler",
        StandardScaler())
])

### Categorical preprocessing:
1. Fill missing values with most frequent category
2. Convert categories into numbers

In [15]:
cat_pipeline = Pipeline([
    ("imputer",
        SimpleImputer(strategy="most_frequent")),

    ("encoder",
        OneHotEncoder(handle_unknown="ignore"))
])

### Apply transformations to respective columns

In [16]:
preprocessor = ColumnTransformer([
    ("num",
        num_pipeline,
        numerical_cols),

    ("cat",
        cat_pipeline,
        categorical_cols)
])

In [18]:
def get_preprocessor(num_pipeline, cat_pipeline, numerical_cols, categorical_cols):
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, numerical_cols),
        ("cat", cat_pipeline, categorical_cols)
    ])
    return preprocessor